# Can we predict late delivery?
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-split walk-forward CV on training rows, selection by mean CV ROC AUC.

**Feature decision:** `claim_paid_eur` is compensation paid on a claim settled *after* delivery, so it is not known at prediction time (it is ~90% positive when >0, 0% late otherwise: outcome leakage). It is excluded, besides `consignment` (identifier) and `dispatched_on` (split column, per protocol).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
df = pd.read_csv('data.csv', parse_dates=['dispatched_on'])
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx)==1610
train = df.drop(index=hold_idx).sort_values('dispatched_on', kind='stable')
hold = df.loc[hold_idx]
print(len(train), len(hold), train.dispatched_on.max(), hold.dispatched_on.min())

In [ ]:
TARGET='late_delivery'
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT   # claim_paid_eur excluded: recorded after the outcome
Xtr, ytr = train[FEATURES], train[TARGET]
print(ytr.mean(), hold[TARGET].mean())

All candidates are named before any is scored.

In [ ]:
def pre(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
candidates = {
 'baseline_prior': make_pipeline(pre(False), DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(pre(True), LogisticRegression(C=1.0, max_iter=1000)),
 'logreg_C0.1': make_pipeline(pre(True), LogisticRegression(C=0.1, max_iter=1000)),
 'random_forest': make_pipeline(pre(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)),
 'hist_gb': make_pipeline(pre(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
list(candidates)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
folds = [(tr, va) for tr, va in tscv.split(Xtr)]
cv_scores = {}
for name, model in candidates.items():
    s=[]
    for tr, va in folds:
        m = clone(model).fit(Xtr.iloc[tr], ytr.iloc[tr])   # preprocessing fitted inside the fold
        s.append(roc_auc_score(ytr.iloc[va], m.predict_proba(Xtr.iloc[va])[:,1]))
    cv_scores[name]=s
cv = pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv

In [ ]:
chosen = cv['mean'].idxmax(); print('chosen:', chosen, cv.loc[chosen,'mean'])

In [ ]:
final = clone(candidates[chosen]).fit(Xtr, ytr)
p = final.predict_proba(hold[FEATURES])[:,1]
holdout_scores = {'roc_auc': roc_auc_score(hold[TARGET], p), 'average_precision': average_precision_score(hold[TARGET], p)}
print(holdout_scores, 'base rate', hold[TARGET].mean())   # scored once

In [ ]:
lab = lambda ix: [int(i) for i in ix]
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [{'train_rows': lab(train.index[tr]), 'validation_rows': lab(train.index[va])} for tr,va in folds],
 'cv_scores': {k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen': chosen,
 'holdout_rows': lab(hold_idx),
 'holdout_scores': {k:float(v) for k,v in holdout_scores.items()},
}
json.dump(evidence, open('evidence.json','w'))